# Airport Multimodal Chatbot — Analytics & Optimisation

**Module:** Chatbot Analytics and Optimization · MSc Artificial Intelligence · BSBI
**Author:** Utkarsh Verma · **Date:** September 2026

This notebook accompanies my report *"Optimising Intelligent Chatbot Performance through Advanced Data Analytics"*. It reproduces every analytic finding cited in the report and complements the live dashboard hosted at:

**Live dashboard:** https://huggingface.co/spaces/utkarsh141verma007/airport-chatbot-analytics
**Source chatbot repo:** https://github.com/Utkarsh132/airport_chatbot

**How to use:** *Runtime → Run all*. All data is fetched from the public repo; nothing else needs to be uploaded.

---


## 1. Setup

Install + import the libraries used across the analysis.


In [ ]:
!pip install --quiet pandas numpy scikit-learn matplotlib seaborn

import io, json
from urllib.request import urlopen
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
plt.rcParams.update({"figure.facecolor":"white", "axes.facecolor":"white"})
print("Environment ready.")


## 2. Load simulated conversation logs

The logs are generated once by `simulate_logs.py` (in the project repo) and committed as CSV so the analytics are fully reproducible from the notebook alone.

- **Grounded in real assets**: 38 knowledge-base records and 81 real passenger utterances from the underlying chatbot's `data/` folder.
- **Persona-aware simulation**: five personas (business, leisure, family, transit, first_time), each with its own intent-mix, confusion pattern, latency profile and CSAT curve.
- **Journey co-occurrence links**: e.g. *find_gate → boarding_time → security* fires more often than chance, matching real airport wayfinding patterns.


In [ ]:
# Public raw URLs — no auth needed
BASE = "https://raw.githubusercontent.com/Utkarsh132/airport_chatbot/main/chatbot_analysis/outputs"

users    = pd.read_csv(f"{BASE}/logs/users.csv")
sessions = pd.read_csv(f"{BASE}/logs/sessions.csv")
turns    = pd.read_csv(f"{BASE}/logs/conversation_logs.csv",
                       parse_dates=["timestamp"])

print(f"users:    {len(users):>5}")
print(f"sessions: {len(sessions):>5}")
print(f"turns:    {len(turns):>5}")
turns.head(3)


## 3. Exploratory Data Analysis

Descriptive statistics for the log window (personas, channels, response times, CSAT).


In [ ]:
eda = {
    "n_users":    users["user_id"].nunique(),
    "n_sessions": sessions["session_id"].nunique(),
    "n_turns":    len(turns),
    "avg_turns_per_session": round(turns.groupby("session_id").size().mean(), 2),
    "unique_intents_observed":  turns["true_intent"].nunique(),
    "response_time_ms_median":  int(turns["response_time_ms"].median()),
    "response_time_ms_p95":     int(turns["response_time_ms"].quantile(0.95)),
    "ttfr_ms_median":           int(turns["time_to_first_response_ms"].median()),
    "csat_mean":                round(turns["csat"].mean(), 2),
    "fallback_rate_%":          round(turns["is_fallback"].mean()*100, 2),
    "session_completion_%":     round(sessions["completed"].mean()*100, 2),
}
print(json.dumps(eda, indent=2))


In [ ]:
# Persona × channel session mix
pd.crosstab(sessions["persona"], sessions["channel"], margins=True)


## 4. Analytic Area 1 — User segmentation & personalisation

Segment by traveller persona (business, leisure, family, transit, first_time) and by channel (mobile, kiosk, web). We look at CSAT, session length, fallback rate and completion to spot who the bot serves well vs. who it struggles with.


In [ ]:
seg = (sessions.groupby("persona")
       .agg(sessions   = ("session_id","count"),
            csat       = ("csat","mean"),
            length     = ("n_turns","mean"),
            completion = ("completed","mean"))
       .round(3)
       .sort_values("csat", ascending=False))

# Add fallback rate from turn-level data
fb_per = turns.groupby("persona")["is_fallback"].mean().mul(100).round(2)
seg["fallback_%"] = fb_per
seg


In [ ]:
# Visualise: fallback rate by persona
fb = seg["fallback_%"].sort_values()
colors = ["#22c55e" if v<6 else "#f5a524" if v<8 else "#ef4444" for v in fb.values]

fig, ax = plt.subplots(figsize=(7,3.2))
bars = ax.barh(fb.index, fb.values, color=colors)
ax.set_xlabel("Fallback rate (%)")
ax.set_title("Fallback rate by persona — higher = model struggling")
for bar, val in zip(bars, fb.values):
    ax.text(val+0.1, bar.get_y()+bar.get_height()/2, f"{val:.1f}%", va="center")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.show()


**Reading the segmentation.**
- **Business** and **transit** travellers show the highest fallback rates (~9%). Both ask more compressed, time-critical questions with airport-specific jargon (transfer desk, gate change, connection) — this is where the model has the most head-room to improve.
- **First-time** flyers get the fewest fallbacks (~5%) because their questions map cleanly to the beginner-friendly intent set (restroom, information desk, check-in).
- **Family** sits in the middle; the frustration comes from multi-child logistics not covered by KB (family lounge access, unaccompanied minors).


## 5. Analytic Area 2 — Fallback rate + NLP-based detector

Two complementary fallback signals:
1. **Confidence-based** — the classifier's own probability drops below 0.35.
2. **NLP-based** — we scan the *bot's response* (or in this simulation, the utterance ↔ predicted-intent mismatch) for linguistic markers of an unresolved turn: *"don't understand"*, *"can you rephrase"*, *"I'm not sure"*, *"sorry"*.

Comparing the two catches cases where the classifier is over-confident on an out-of-scope query — a common failure mode in production bots.


In [ ]:
import re

# ------- rule-based NLP fallback detector -------------------------------
FALLBACK_PHRASES = [
    r"\bdon'?t (?:know|understand)\b",
    r"\bcan you (?:rephrase|repeat|clarify)\b",
    r"\bi'?m (?:not sure|sorry)\b",
    r"\bsorry(?:,| I)\b",
    r"\bout of scope\b",
    r"\bunable to (?:help|answer)\b",
]
pattern = re.compile("|".join(FALLBACK_PHRASES), flags=re.I)

def linguistic_fallback(text: str) -> bool:
    return bool(pattern.search(text or ""))

# demo on a handful of synthetic bot responses
demo = [
    "I don't understand your question.",
    "Sorry, I'm not sure I can help with that.",
    "Please rephrase your question.",
    "Your gate is B12."
]
for u in demo:
    print(f"  fallback={linguistic_fallback(u)!s:5} :: {u}")


In [ ]:
# Combined fallback detection on the log
CONFIDENCE_THRESH = 0.35
turns["nlp_flag"]        = turns["utterance"].apply(linguistic_fallback)
turns["confidence_flag"] = turns["confidence"] < CONFIDENCE_THRESH
turns["combined_flag"]   = turns["nlp_flag"] | turns["confidence_flag"] | \
                           (turns["predicted_intent"] == "FALLBACK")

report = {
    "confidence_based_flags": int(turns["confidence_flag"].sum()),
    "predicted_intent_flags": int((turns["predicted_intent"] == "FALLBACK").sum()),
    "linguistic_marker_flags": int(turns["nlp_flag"].sum()),
    "combined_nlp_flags":     int(turns["combined_flag"].sum()),
    "logged_is_fallback":     int(turns["is_fallback"].sum()),
    "overall_fallback_rate_%":round(turns["combined_flag"].mean()*100, 2),
}
print(json.dumps(report, indent=2))


**Interpretation.**
- Overall fallback rate lands at **~8%**, comfortably inside the healthy 5–15 % industry band for domain-specific bots ([Rasa 2023 benchmark](https://rasa.com/blog/measuring-chatbot-success/)).
- The confidence-based and prediction-based detectors are almost tied — evidence the classifier's uncertainty is well-calibrated on this dataset.
- The linguistic detector fires 0 times on user utterances (as expected — passengers don't produce fallback language) but the same regex would catch bot-side "sorry, I can't help" responses in a real production log; the code is drop-in ready.


In [ ]:
# Which intents drive the fallbacks?
fb_int = (turns.assign(intent=turns["true_intent"])
          .groupby("intent")["is_fallback"].mean()
          .mul(100).round(1)
          .sort_values(ascending=False).head(10))
fb_int.to_frame("fallback_%")


## 6. Analytic Area 3 — Intent recognition accuracy + confusion matrix

We evaluate on the subset of turns where the true intent is *classifiable* (i.e. a real intent, excluding "unknown"). This isolates model-quality issues from out-of-scope traffic.


In [ ]:
classifiable = turns[turns["true_intent"] != "unknown"].copy()

y_true = classifiable["true_intent"].values
y_pred = classifiable["predicted_intent"].values

acc  = accuracy_score(y_true, y_pred)
f1_m = f1_score(y_true, y_pred, average="macro", zero_division=0)
f1_w = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"Accuracy        : {acc*100:.2f}%")
print(f"F1 (macro)      : {f1_m:.3f}")
print(f"F1 (weighted)   : {f1_w:.3f}")
print(f"Classifiable n  : {len(classifiable)}")


In [ ]:
# Confusion matrix on the 15 most frequent intents
top_intents = classifiable["true_intent"].value_counts().head(15).index.tolist()
mask = classifiable["true_intent"].isin(top_intents) & classifiable["predicted_intent"].isin(top_intents)
cm = confusion_matrix(
    classifiable.loc[mask, "true_intent"],
    classifiable.loc[mask, "predicted_intent"],
    labels=top_intents,
)
cm_df = pd.DataFrame(cm, index=top_intents, columns=top_intents)

fig, ax = plt.subplots(figsize=(8,7))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
            linewidths=0.4, linecolor="#e6edf7")
ax.set_xlabel("Predicted intent"); ax.set_ylabel("True intent")
ax.set_title("Confusion matrix — top 15 intents")
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()


**Diagonal reading.** Clean diagonal on most intents. The heaviest off-diagonal errors are:

| True intent | Predicted (wrong) | # confusions |
|---|---|---|
| transfer_desk | information_desk | 22 |
| transfer_desk | check_in | 9 |
| transit_connection | find_gate | 9 |
| transit_connection | security | 5 |

These are *semantic* errors — the bot conflates "transfer" with generic help. Fix in Part 3: retrain on augmented transfer-desk paraphrases, add distinctive keyword features ("connecting flight", "layover", "transit").


## 7. Analytic Area 4 — Intent co-occurrence (Support / Confidence / Lift)

Treat each *session* as a basket of intents. Then for every ordered pair `(A → B)` compute:

- **Support(A→B)** = P(A ∧ B) — how often both appear together
- **Confidence(A→B)** = P(B | A) — of sessions containing A, what fraction also contain B
- **Lift(A→B)** = P(B | A) / P(B) — how much A boosts the odds of B (>1 = positive association)

We filter to rules with support ≥ 2 % and rank by lift. These are the strongest passenger-journey patterns.


In [ ]:
from itertools import permutations

# Baskets: unique intents per session (excluding unknown/fallback)
baskets = (turns[~turns["true_intent"].isin(["unknown","FALLBACK"])]
           .groupby("session_id")["true_intent"]
           .apply(lambda s: set(s.dropna())))
N = len(baskets)
support_one = pd.Series({i: sum(i in b for b in baskets) / N
                        for i in set().union(*baskets)})

rules = []
top_items = support_one[support_one >= 0.05].index  # only common enough intents
for a, b in permutations(top_items, 2):
    joint = sum((a in bkt) and (b in bkt) for bkt in baskets) / N
    if joint < 0.02:
        continue
    conf = joint / support_one[a]
    lift = conf / support_one[b]
    rules.append((a, b, joint, conf, lift))

rules_df = (pd.DataFrame(rules, columns=["antecedent","consequent","support","confidence","lift"])
              .sort_values("lift", ascending=False)
              .round(3)
              .head(15))
rules_df


**Journey rules that pop out.**
- **wifi → charging_station (lift 3.06)** — a device-hungry passenger cluster; we should surface both in one card.
- **shopping → currency_exchange (lift 2.44)** — the retail-therapy transit cluster; bundle in a "landside amenities" quick-reply.
- **restroom ↔ restaurant (lift 2.22)** — bidirectional; both are "physiological stop" intents.
- **information_desk → restaurant (lift 2.20)** — lost passengers who then ask about food. If the bot answers *information_desk* well, we should proactively offer *restaurant* as a next-best suggestion.

These rules feed the personalisation strategy in Part 3.


## 8. Response-time metrics

Latency percentiles + a CDF. Anything above 3 s at p95 for a text bot signals infrastructure pressure ([Nielsen usability heuristic on response time](https://www.nngroup.com/articles/response-times-3-important-limits/)).


In [ ]:
rt = turns["response_time_ms"].dropna()
print(f"median:  {int(rt.median())} ms")
print(f"p95:     {int(rt.quantile(0.95))} ms")
print(f"p99:     {int(rt.quantile(0.99))} ms")

fig, ax = plt.subplots(figsize=(8,3.2))
sorted_rt = np.sort(rt.values)
cdf = np.arange(1, len(sorted_rt)+1) / len(sorted_rt) * 100
ax.plot(sorted_rt, cdf, color="#22d3ee", linewidth=2)
ax.axvline(rt.median(), ls="--", color="#8fa2c8", label=f"p50 {int(rt.median())} ms")
ax.axvline(rt.quantile(0.95), ls="--", color="#ef4444", label=f"p95 {int(rt.quantile(0.95))} ms")
ax.set_xlabel("Response time (ms)"); ax.set_ylabel("% of turns ≤ t")
ax.set_title("Response-time CDF"); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()


## 9. Lifetime-value (LTV) proxy by persona

LTV for a free chatbot isn't dollars — it's engagement quality. We use:

$$\text{LTV} = \underbrace{\text{sessions per user}}_{\text{repeat use}} \times \underbrace{\text{avg CSAT}}_{\text{quality}} \times \underbrace{\text{completion rate}}_{\text{task success}}$$

Higher = the persona keeps coming back, is happy, and finishes what they start.


In [ ]:
user_pers = users.set_index("user_id")["persona"]
sess_per_user = sessions.groupby("user_id").size()

ltv = (pd.DataFrame({"persona": user_pers, "sessions_per_user": sess_per_user})
       .dropna()
       .groupby("persona")
       .agg(sessions_per_user=("sessions_per_user","mean"))
       .join(sessions.groupby("persona")
                     .agg(csat=("csat","mean"),
                          completion=("completed","mean")))
       .assign(ltv_score=lambda d: d["sessions_per_user"] * d["csat"] * d["completion"])
       .round(3)
       .sort_values("ltv_score", ascending=False))
ltv


**LTV winners.** Business travellers score highest (repeat use + still-decent CSAT even when fallback rate is elevated). Transit and family follow. First-time and leisure — although easier for the bot to answer — visit less often, dragging LTV.

**Strategic implication.** Investment in transfer-desk / gate-change accuracy improves the highest-LTV segment. Even a 2-point accuracy lift on business queries yields disproportionate engagement gains.


## 10. Frustration analysis

We flag a session as frustrated when any turn shows a **repetition marker** (user repeats the same question, "for the third time", "I already asked"). Cheap heuristic, high recall in the log.


In [ ]:
# Pull an example frustration session for the report
frust_ex = pd.read_csv(f"{BASE}/analytics/tables/frustration_examples.csv")
sample_sid = frust_ex["session_id"].value_counts().index[0]
frust_ex[frust_ex["session_id"] == sample_sid][["turn_index","utterance","true_intent","predicted_intent","csat"]]


## 11. Dashboard — Departure-Board Analytics

All of the above KPIs are unified in a single dashboard designed around the airport departure-board (FIDS) aesthetic. Static PNG rendered below; **interactive Plotly.js version** with persona & channel filters is hosted at:

**https://huggingface.co/spaces/utkarsh141verma007/airport-chatbot-analytics**


In [ ]:
from IPython.display import Image, display
display(Image(url=f"{BASE}/dashboard/dashboard.png"))


## 12. Summary of findings

| Metric | Value | Verdict |
|---|---|---|
| Intent accuracy (classifiable) | **81 %** | inside industry-healthy band (75–85 %) |
| Fallback rate | **7.9 %** | inside healthy 5–15 % band |
| Avg CSAT | **3.88 / 5** | above domain median (3.5) |
| Session completion | **71 %** | strong, ceiling ~85 % |
| Median latency | **889 ms** | comfortably below 1 s human-attention threshold |
| Frustration incidence | **19 %** of sessions | actionable — focus on transfer-desk + transit intents |

**Top three optimisation priorities** (see report Part 3):
1. Augment training for *transfer_desk* / *transit_connection* — accounts for the largest misclassification cluster.
2. Personalise the follow-up quick-replies using the co-occurrence rules (wifi→charging, shopping→currency).
3. Add a two-strike escalation to human handoff when repetition markers are detected within a session.
